In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input
import xgboost as xgb
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

In [4]:
# Step 1: Data Preparation

# Load and preprocess the data
mitbih_train = pd.read_csv('mitbih_train.csv', header=None)
mitbih_test = pd.read_csv('mitbih_test.csv', header=None)

# Separate features and labels
X_train = mitbih_train.iloc[:, :-1].values
y_train = mitbih_train.iloc[:, -1].values

X_test = mitbih_test.iloc[:, :-1].values
y_test = mitbih_test.iloc[:, -1].values

# Initial train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

# Apply SMOTE to balance the classes in the training data
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Standardize data
scaler = StandardScaler()
X_train_smote = scaler.fit_transform(X_train_smote)
X_test = scaler.transform(X_test)

# Reshape the original data for Conv2D
X_train_reshaped = X_train_smote.reshape(-1, 17, 11, 1)
X_test_reshaped = X_test.reshape(-1, 17, 11, 1)

In [5]:
# Step 2: Build and Train the CNN Model
# CNN Model Definition
def create_cnn_model():
    inputs = Input(shape=(17, 11, 1))
    
    # Convolutional layers
    x = Conv2D(32, kernel_size=(3, 3), activation='relu')(inputs)
    x = MaxPooling2D(pool_size=(2, 2))(x)
    x = Dropout(0.25)(x)
    
    x = Conv2D(64, kernel_size=(2, 2), activation='relu')(x)
    x = MaxPooling2D(pool_size=(2, 2))(x)
    x = Dropout(0.25)(x)
    
    # Flatten the convolutional output
    x = Flatten()(x)
    
    # Dense layers
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.5)(x)
    
    # Output layer
    outputs = Dense(5, activation='softmax')(x)  # 5 classes for MIT-BIH
    
    # Build the model
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

cnn_model = create_cnn_model()

# Calculate class weights
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_smote), y=y_train_smote)
class_weight_dict = {i: class_weights[i] for i in range(len(class_weights))}

# Train the CNN model
cnn_model.fit(X_train_reshaped, y_train_smote,
              epochs=20,
              batch_size=256,
              validation_data=(X_test_reshaped, y_test),
              class_weight=class_weight_dict,
              callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)])


Epoch 1/20
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 17s 13ms/step - accuracy: 0.6608 - loss: 0.8651 - val_accuracy: 0.8535 - val_loss: 0.5494
Epoch 2/20
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - accuracy: 0.8564 - loss: 0.3940 - val_accuracy: 0.8754 - val_loss: 0.4124
Epoch 3/20
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - accuracy: 0.8859 - loss: 0.3197 - val_accuracy: 0.8923 - val_loss: 0.3384
Epoch 4/20
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - accuracy: 0.9004 - loss: 0.2813 - val_accuracy: 0.9063 - val_loss: 0.2869
Epoch 5/20
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 14s 12ms/step - accuracy: 0.9111 - loss: 0.2533 - val_accuracy: 0.9142 - val_loss: 0.2636
Epoch 6/20
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 14s 12ms/step - accuracy: 0.9171 - loss: 0.2398 - val_accuracy: 0.9282 - val_loss: 0.2123
Epoch 7/20
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - accuracy: 0.9214 - loss: 0.2260 - val_accuracy: 0.9262 - val_loss: 0.2207
Epoch 8/20
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - accuracy: 0.9243 -

In [6]:
# Step 3: Extract Features from CNN Model

# Extract features from the trained CNN model (from the second-to-last layer)
feature_extractor = Model(inputs=cnn_model.input, outputs=cnn_model.layers[-3].output)

# Get the features for both training and test sets
X_train_features = feature_extractor.predict(X_train_reshaped)
X_test_features = feature_extractor.predict(X_test_reshaped)


9059/9059 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step
548/548 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [7]:
# Step 4: Train XGBoost on Extracted Features

# Define the XGBoost model
xgb_model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=5,
    eval_metric='mlogloss',
    use_label_encoder=False
)

# Train the XGBoost model on the extracted features
xgb_model.fit(X_train_features, y_train_smote)

c:\Users\asena\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [15:06:56] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_class=5, num_parallel_tree=None, ...)

In [8]:
# Step 5: Evaluate the XGBoost Model

# Make predictions and generate a classification report
y_pred_xgb = xgb_model.predict(X_test_features)

classification_report_str = classification_report(y_test, y_pred_xgb, target_names=['N', 'S', 'V', 'F', 'Q'])
print("Classification Report:\n", classification_report_str)

Classification Report:
               precision    recall  f1-score   support

           N       0.99      0.96      0.98     14494
           S       0.52      0.84      0.64       445
           V       0.89      0.94      0.91      1158
           F       0.53      0.81      0.64       128
           Q       0.97      0.99      0.98      1286

    accuracy                           0.96     17511
   macro avg       0.78      0.91      0.83     17511
weighted avg       0.97      0.96      0.96     17511

